# LatentSync on Kaggle T4 x2 — 通用双版本推理

**硬件**：`GPU T4 x2` = 两块独立的 Tesla T4（每块约 15GB 可用显存），不是一张 32GB 卡。

**本 Notebook 是通用的**：代码里不写任何账号、数据集名或素材文件名。素材从 `/kaggle/input` 自动发现，也可以由仓库里的 `run.py` 传入一份清单，所以「换人、换机、换 token、换素材」都不用改这个文件。

**跑什么**：LatentSync **1.5 @ 256**，一条视频 + 一段音频，出两个版本：

| 变体 | 输入处理 | 输出 |
| --- | --- | --- |
| `fullbody` | 长边缩到 1440（保持比例） | `/kaggle/working/fullbody/video_out.mp4` |
| `closeup` | 按原片比例裁头肩区域，再缩到短边 1280 | `/kaggle/working/closeup/video_out.mp4` |

环境（Python 3.10 虚拟环境 + 权重）只搭一次，两个变体复用。每个变体各自写 `run.log` / `result.json`，最后汇总到 `summary.json`。

**为什么单独建虚拟环境**：Kaggle 当前镜像用 Python 3.13 + torch 2.11，而 LatentSync 官方钉的是 Python 3.10 + torch 2.5.1（mediapipe 0.10.11、opencv 4.9、numpy 1.26 都没有 3.13 的 wheel）。所以这里用 `uv` 建一个 Python 3.10 环境装官方依赖，Notebook 自身仍跑在 3.13 上。

**两个关键取舍**

1. LatentSync **1.6 @ 512 推理约需 18GB**，单张 T4 放不下；两卡之间没有 NVLink，官方 pipeline 也不支持跨卡拆 UNet，所以这套免费环境选 1.5。
2. 官方 `scripts/inference.py` 在 `compute capability <= 7` 时把精度从 fp16 降成 fp32（T4 是 7.5）。fp32 显存约翻倍、速度明显变慢，因此下面会打一个一行的补丁，让 T4 也能用 fp16。

**手动在网页里跑**：Settings → Accelerator 选 `GPU T4 x2`，Internet 选 `On`，挂载含视频和音频的数据集即可。首次整体约 40–55 分钟（装依赖 + 下 5GB 左右权重 + 两次推理）。

## 1. 输入自检：自动发现视频与音频

优先读数据集里的 `inputs.json`（由 `run.py` 写入，指明用哪两个文件）；没有就按扩展名在 `/kaggle/input` 下自动找。

In [ ]:
import os, subprocess, sys, time, json, threading, shutil, glob

WORK = "/kaggle/working"
SCRATCH = "/kaggle/tmp"
REPO = os.path.join(SCRATCH, "LatentSync")
VENV = os.path.join(SCRATCH, "venv-latentsync")
VENV_PY = os.path.join(VENV, "bin", "python")
os.makedirs(WORK, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)

print("Python:", sys.version.split()[0])
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

VIDEO_EXT = {".mp4", ".mov", ".mkv", ".webm", ".m4v", ".avi"}
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".flac", ".aac", ".ogg"}

INPUT_FILES = []
for _root, _, _files in os.walk("/kaggle/input"):
    for _f in _files:
        INPUT_FILES.append(os.path.join(_root, _f))
INPUT_FILES.sort()

print(f"挂载的输入文件（{len(INPUT_FILES)} 个）:")
for _p in INPUT_FILES:
    print(f"  {os.path.getsize(_p)/1024**2:9.2f} MB  {_p}")

def _json_input(name):
    for _p in INPUT_FILES:
        if os.path.basename(_p) == name:
            try:
                return json.load(open(_p, encoding="utf-8"))
            except Exception as exc:
                print(f"读取 {name} 失败：{exc}")
    return None

manifest = _json_input("inputs.json") or {}
run_cfg = _json_input("run-config.json") or {}
print("inputs.json:", manifest)
print("run-config.json:", run_cfg)

def _by_name(name):
    for _p in INPUT_FILES:
        if os.path.basename(_p) == name:
            return _p
    raise FileNotFoundError(f"数据集里没有声明过的文件 {name}，请检查 inputs.json")

def _pick(exts, prefer, kind):
    if prefer:
        for _p in INPUT_FILES:
            if os.path.splitext(_p)[1].lower() in exts and prefer in os.path.basename(_p):
                return _p
    cands = [_p for _p in INPUT_FILES if os.path.splitext(_p)[1].lower() in exts]
    if not cands:
        raise FileNotFoundError(f"没有找到{kind}文件（支持 {sorted(exts)}）：请挂载数据集，或把文件放进仓库的 inputs/ 后用 run.py 上传")
    if len(cands) > 1:
        print(f"提示：找到多个{kind}候选，取第一个；需要用别的可设置环境变量或由 run.py 指定")
        print("  候选:", [os.path.basename(c) for c in cands])
    return cands[0]

USER_VIDEO = _by_name(manifest["video"]) if manifest.get("video") else _pick(VIDEO_EXT, os.environ.get("VIDEO_NAME"), "视频")
USER_AUDIO = _by_name(manifest["audio"]) if manifest.get("audio") else _pick(AUDIO_EXT, os.environ.get("AUDIO_NAME"), "音频")
print("VIDEO:", USER_VIDEO)
print("AUDIO:", USER_AUDIO)


## 2. 拉取代码 + 系统依赖（ffmpeg / libGL）

In [ ]:
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/bytedance/LatentSync.git", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

if subprocess.run(["bash", "-lc", "command -v ffmpeg"], capture_output=True).returncode != 0:
    subprocess.run(["bash", "-lc",
                    "apt-get -qq update && apt-get -qq install -y ffmpeg libgl1 libglib2.0-0 >/dev/null 2>&1"],
                   check=False)
print(subprocess.run(["bash", "-lc", "ffmpeg -version | head -1"],
                     capture_output=True, text=True).stdout.strip())

## 3. 建 Python 3.10 虚拟环境并安装官方依赖

用 `uv` 拉起独立 Python 3.10，再按官方 `requirements.txt` 安装（含 torch 2.5.1 + cu121、torchvision、insightface、mediapipe 等）。这一步最耗时。

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", "uv"], check=True)
UV = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
print("uv:", UV)

env = dict(os.environ, UV_HTTP_TIMEOUT="600")

if not os.path.exists(VENV_PY):
    r = subprocess.run([UV, "venv", "--python", "3.10", VENV], capture_output=True, text=True, env=env)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if r.returncode != 0:
        raise SystemExit("创建 Python 3.10 环境失败")

t0 = time.time()
r = subprocess.run(
    [UV, "pip", "install", "--python", VENV_PY,
     "torch==2.5.1", "torchvision==0.20.1",
     "-r", "requirements.txt"],
    cwd=REPO, capture_output=True, text=True, env=env)
print(f"exit={r.returncode}  {time.time()-t0:.0f}s", flush=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print(r.stderr[-8000:])
    raise SystemExit("依赖安装失败，请查看上面的安装日志")
print("dependencies ready")

## 4. 确认装好的是 2.5.1 + 两张 T4 可见

In [ ]:
probe = (
    "import torch;"
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count());"
    "[print(i, torch.cuda.get_device_properties(i).name,"
    " round(torch.cuda.get_device_properties(i).total_memory/1024**3, 1), 'GB')"
    " for i in range(torch.cuda.device_count())]"
)
r = subprocess.run([VENV_PY, "-c", probe], capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit("torch 检查失败")

## 5. 下载推理权重（LatentSync 1.5：UNet + Whisper tiny，约 5GB）

In [ ]:
CKPT = os.path.join(REPO, "checkpoints")
os.makedirs(os.path.join(CKPT, "whisper"), exist_ok=True)

dl = (
    "import os, time\n"
    "from huggingface_hub import hf_hub_download\n"
    f"CKPT = {CKPT!r}\n"
    "for repo_id, filename in [('ByteDance/LatentSync-1.5', 'latentsync_unet.pt'),"
    " ('ByteDance/LatentSync-1.5', 'whisper/tiny.pt')]:\n"
    "    t = time.time()\n"
    "    p = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=CKPT)\n"
    "    print(f'{filename} -> {p}  {os.path.getsize(p)/1024**2:.0f} MB  {time.time()-t:.0f}s', flush=True)\n"
)
r = subprocess.run([VENV_PY, "-c", dl], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit("权重下载失败")

## 6. 给 T4 打开 fp16

官方判断是 `get_device_capability()[0] > 7`，T4 的 7.5 被判成 fp32。改成 `>= 7` 后 T4 走 fp16：显存减半、速度明显提升。

In [ ]:
infer_py = os.path.join(REPO, "scripts", "inference.py")
src = open(infer_py).read()
patched = src.replace("get_device_capability()[0] > 7", "get_device_capability()[0] >= 7")
if patched != src:
    open(infer_py, "w").write(patched)
print("fp16 patch applied:", patched != src)
# 人脸占画面比例小（全身镜头）时，512 的检测输入会漏检；放大到 1024
det_py = os.path.join(REPO, "latentsync", "utils", "face_detector.py")
src = open(det_py).read()
patched = src.replace("INSIGHTFACE_DETECT_SIZE = 512", "INSIGHTFACE_DETECT_SIZE = 1024")
if patched != src:
    open(det_py, "w").write(patched)
print("detect size patch applied:", patched != src)

## 7. 预处理：全身 + 半身两份工作副本

`LipsyncPipeline` 会把整段视频一次性解码成张量：2160×3840 × 1272 帧约需 120GB 内存，29GB 的 Kaggle 机器必然 OOM，所以两个变体都先重编码成小尺寸工作副本（不含音轨，音频单独由 `--audio_path` 提供）。

- 全身：长边 1440，保持比例
- 半身：默认按原片比例裁头肩（宽 37%、高 37%、左上偏移 23%/23%），再缩到高 1280；可用 `run-config.json` 或环境变量覆盖
- 只跑其中一个：`VARIANTS_JSON` / `RUN_ONLY`，或 `run.py --only fullbody`

In [ ]:
MAX_SIDE = int(os.environ.get("MAX_SIDE", run_cfg.get("max_side", 1440)))
CROP_PCT = os.environ.get("CLOSEUP_CROP_PCT", run_cfg.get("crop_pct", "37:37:23:23"))
CROP_PX = os.environ.get("CLOSEUP_CROP", run_cfg.get("crop_px", ""))
CROP_OUT = str(os.environ.get("CLOSEUP_OUT", run_cfg.get("crop_out", "auto")))

DEFAULT_VARIANTS = [
    {"name": "fullbody", "mode": "scale", "label": "全身"},
    {"name": "closeup", "mode": "crop", "label": "半身"},
]
VARIANTS = run_cfg.get("variants") or DEFAULT_VARIANTS
if VARIANTS and isinstance(VARIANTS[0], str):
    VARIANTS = [v for v in DEFAULT_VARIANTS if v["name"] in VARIANTS]
only = [s.strip() for s in os.environ.get("RUN_ONLY", "").split(",") if s.strip()]
if only:
    VARIANTS = [v for v in VARIANTS if v["name"] in only]
if not VARIANTS:
    raise SystemExit("没有要跑的变体：检查 run-config.json / RUN_ONLY")
print("变体:", [v["name"] for v in VARIANTS])

def _even(x):
    return max(2, int(round(float(x) / 2.0)) * 2)

_p = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                     "-show_entries", "stream=width,height", "-of", "csv=p=0:s=x", USER_VIDEO],
                    capture_output=True, text=True)
sw, sh = [int(x) for x in _p.stdout.strip().split("x")]
print(f"源片: {sw}x{sh}")

_a = subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                     "format=duration:stream=codec_name,sample_rate,channels",
                     "-of", "json", USER_AUDIO], capture_output=True, text=True)
print("音频:", _a.stdout.strip())

prepared = {}
for v in VARIANTS:
    dst = os.path.join(SCRATCH, f"prepared_{v['name']}.mp4")
    t = time.time()
    vf, crf = None, "18"
    if v["mode"] == "crop":
        if CROP_PX:
            cw, ch, cx, cy = [int(x) for x in CROP_PX.split(":")]
        else:
            pw, ph, px, py = [float(x) for x in CROP_PCT.split(":")]
            cw, ch = _even(sw * pw / 100), _even(sh * ph / 100)
            cx, cy = _even(sw * px / 100), _even(sh * py / 100)
        if cw > sw or ch > sh:
            raise SystemExit(f"裁切尺寸 {cw}x{ch} 超出源片 {sw}x{sh}，请调整 CLOSEUP_CROP_PCT")
        cx = max(0, min(cx, sw - cw))
        cy = max(0, min(cy, sh - ch))
        if CROP_OUT.lower() == "auto":
            ow, oh = _even(cw * 1280.0 / ch), 1280
        else:
            ow, oh = [int(x) for x in CROP_OUT.lower().split("x")]
        vf = f"crop={cw}:{ch}:{cx}:{cy},scale={ow}:{oh}"
        crf = "17"
        print(f"{v['label']}：裁 {cw}x{ch}+{cx}+{cy} -> {ow}x{oh}")
    else:
        if max(sw, sh) > MAX_SIDE:
            s = MAX_SIDE / float(max(sw, sh))
            ow, oh = _even(sw * s), _even(sh * s)
            vf = f"scale={ow}:{oh}"
        else:
            ow, oh = sw, sh
    if vf is None:
        shutil.copy(USER_VIDEO, dst)
    else:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", USER_VIDEO,
                        "-vf", vf, "-c:v", "libx264", "-crf", crf, "-preset", "veryfast",
                        "-pix_fmt", "yuv420p", "-an", dst], check=True)
    prepared[v["name"]] = {"path": dst, "width": ow, "height": oh, "label": v["label"]}
    print(f"{v['label']}: {ow}x{oh}  {os.path.getsize(dst)/1024**2:.1f} MB  {time.time()-t:.0f}s")


## 8. 人脸预检（每个变体都查）

全身镜头里人脸可能只占画面 8% 左右，检测器有漏检风险。每个变体先抽样 5 帧确认能检出，避免跑满 20 分钟才在第 0 帧报 `Face not detected`。

In [ ]:
import textwrap
pre = textwrap.dedent('''
import sys, os, json, cv2
sys.path.insert(0, ".")
from latentsync.utils.face_detector import FaceDetector

video = os.environ["PREFLIGHT_VIDEO"]
det = FaceDetector(device="cuda")
cap = cv2.VideoCapture(video)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = {"video": video, "size": [w, h], "frames": total, "samples": []}
for idx in sorted({0, total // 4, total // 2, 3 * total // 4, total - 2}):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, fr = cap.read()
    if not ok:
        continue
    raw = [[round(float(f.det_score), 3), [int(v) for v in f.bbox]] for f in det.app.get(fr)]
    bbox = det(fr)[0]
    out["samples"].append({
        "frame": int(idx),
        "bbox": [int(v) for v in bbox] if bbox is not None else None,
        "raw": raw,
    })
print("PREFLIGHT_JSON=" + json.dumps(out))
''')
open(os.path.join(SCRATCH, "preflight.py"), "w").write(pre)

preflight = {}
for name, info in prepared.items():
    r = subprocess.run([VENV_PY, os.path.join(SCRATCH, "preflight.py")], cwd=REPO,
                       capture_output=True, text=True,
                       env=dict(os.environ, MPLBACKEND="agg", PREFLIGHT_VIDEO=info["path"]))
    line = [l for l in r.stdout.splitlines() if l.startswith("PREFLIGHT_JSON=")]
    if not line:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise SystemExit(f"{name} 人脸预检执行失败")
    check = json.loads(line[-1].split("=", 1)[1])
    preflight[name] = check
    missing = [s["frame"] for s in check["samples"] if s["bbox"] is None]
    print(f"{name}: {check['size']} 共 {check['frames']} 帧，抽检缺失帧={missing}")
    print(json.dumps(check["samples"], ensure_ascii=False))
    if missing or not check["samples"]:
        raise SystemExit(f"{name} 这些人脸抽检帧检测不到：{missing}；建议调整裁切比例或换近景素材")
print("人脸预检通过")

## 9. 推理：两个变体（各 20 步 + DeepCache）

每个变体单独输出目录，互不覆盖；音频用第一格发现的那一段。

In [ ]:
import threading

TEMP_ROOT = os.path.join(SCRATCH, "temp")
RUN_ENV = dict(os.environ, MPLBACKEND="agg", PYTHONUNBUFFERED="1")
RUN_ENV.pop("DISPLAY", None)

results = {}
for name, info in prepared.items():
    out_dir = os.path.join(WORK, name)
    os.makedirs(out_dir, exist_ok=True)
    out_video = os.path.join(out_dir, "video_out.mp4")
    temp = os.path.join(TEMP_ROOT, name)
    shutil.rmtree(temp, ignore_errors=True)
    os.makedirs(temp, exist_ok=True)

    cmd = [
        VENV_PY, "-m", "scripts.inference",
        "--unet_config_path", "configs/unet/stage2.yaml",
        "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
        "--inference_steps", "20",
        "--guidance_scale", "1.5",
        "--enable_deepcache",
        "--video_path", info["path"],
        "--audio_path", USER_AUDIO,
        "--video_out_path", out_video,
        "--temp_dir", temp,
    ]

    peak = {"mb": 0}
    stop = {"flag": False}

    def _watch(peak=peak, stop=stop):
        while not stop["flag"]:
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"], text=True)
                vals = [int(x) for x in out.split() if x.strip().isdigit()]
                if vals:
                    peak["mb"] = max(peak["mb"], max(vals))
            except Exception:
                pass
            time.sleep(5)

    print(f"=== 开始推理 {name} ({info['width']}x{info['height']}) ===", flush=True)
    threading.Thread(target=_watch, daemon=True).start()
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, env=RUN_ENV)
    elapsed = time.time() - t0
    stop["flag"] = True
    time.sleep(1)

    body = (
        f"variant={name}\n"
        f"video={USER_VIDEO}\n"
        f"audio={USER_AUDIO}\n"
        f"exit_code={r.returncode}\n"
        f"elapsed_s={elapsed:.1f}\n"
        f"peak_vram_mb={peak['mb']}\n\n"
        f"=== STDOUT ===\n{r.stdout}\n\n=== STDERR ===\n{r.stderr}\n"
    )
    open(os.path.join(out_dir, "run.log"), "w").write(body)
    print(body[-4000:], flush=True)

    res = {
        "variant": name,
        "video": USER_VIDEO,
        "audio": USER_AUDIO,
        "exit_code": r.returncode,
        "elapsed_s": round(elapsed, 1),
        "peak_vram_mb": peak["mb"],
        "ok": os.path.isfile(out_video) and os.path.getsize(out_video) > 0,
    }
    if res["ok"]:
        res["size_mb"] = round(os.path.getsize(out_video) / 1024**2, 2)
        pr = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries",
             "format=duration:stream=codec_name,width,height,r_frame_rate",
             "-of", "json", out_video], capture_output=True, text=True)
        res["probe"] = json.loads(pr.stdout or "{}")
    else:
        print(r.stdout[-4000:]); print(r.stderr[-4000:], flush=True)
    json.dump(res, open(os.path.join(out_dir, "result.json"), "w"),
              ensure_ascii=False, indent=2)
    results[name] = res
    print(json.dumps(res, ensure_ascii=False, indent=2), flush=True)

print("推理阶段结束:", {k: v["ok"] for k, v in results.items()})

## 10. 汇总（写入 summary.json）

In [ ]:
summary = {
    "video": USER_VIDEO,
    "audio": USER_AUDIO,
    "variants": results,
}
json.dump(summary, open(os.path.join(WORK, "summary.json"), "w"),
          ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("全部完成" if all(v["ok"] for v in results.values()) else "有变体失败，见上方日志")

## 11. 可选：开一个临时的网页交互入口

Kaggle 不能当长期服务器用（其使用政策禁止 server farming，会话也有 12 小时上限）。如果只是想通过浏览器点着玩，可以启动 Gradio 并套一层 cloudflared 临时隧道，拿到一个几小时内有效的公网地址。

把环境变量 `ENABLE_GRADIO_TUNNEL=1` 后再运行下一格即可；默认跳过，不影响上面的结果。

In [ ]:
if os.environ.get("ENABLE_GRADIO_TUNNEL") == "1":
    import re, urllib.request
    bin_path = "/kaggle/tmp/cloudflared"
    if not os.path.exists(bin_path):
        url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64")
        urllib.request.urlretrieve(url, bin_path)
        os.chmod(bin_path, 0o755)
    print("启动 Gradio（后台）...")
    subprocess.Popen([VENV_PY, "gradio_app.py"], cwd=REPO,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(30)
    tun = subprocess.Popen([bin_path, "tunnel", "--url", "http://127.0.0.1:7860"],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tun.stdout:
        if "trycloudflare.com" in line:
            print("公网地址：", re.search(r"https://[\w.-]+trycloudflare\.com", line).group(0))
            break
    print("隧道进程保持运行中，关闭本 Notebook 会话即失效。")
else:
    print("跳过交互服务。需要时设置 ENABLE_GRADIO_TUNNEL=1 再运行本格。")

## 附：想换 1.6（512）怎么办

1. 下载 `ByteDance/LatentSync-1.6` 的 `latentsync_unet.pt` 覆盖 `checkpoints/`；
2. 把 `--unet_config_path` 换成 `configs/unet/stage2_512.yaml`。

需要约 18GB 显存，单张 T4（15GB）会 OOM；T4 x2 也不能靠官方代码自动拆卡，所以 1.6 建议放到 A100/L40S/4090 这类单卡 ≥24GB 的机器上跑。